In [1]:
import numpy as np

import warnings
from pathlib import Path
import pickle


# from notebooks.data_synchronization_barcode import tsync_barcode
%matplotlib widget

import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import scipy.signal
import pynapple as nap

import pandas as pd
import scipy.io as spio
import os
import tifffile
from totalsync_utils import decode_b64_files
import seaborn as sns
custom_params = {"axes.spines.right": False, "axes.spines.top": False, "figure.figsize": (8, 4)}
sns.set_context("paper")
sns.set_theme(style="ticks", palette="colorblind", font_scale=1.3, rc=custom_params)



In [2]:
def extract_barcode_from_tif(tif_file: str) -> dict:
    tag_structure = {'image_description': 5,
                 'frame_timestamp': 3,
                 'auxTrigger0': 10}  # position of aux tag in tif header

    aux_data = {'frame_n': [],'ts': [], 'value': [] }
    data_reads = []
    #reading the aux line from the tiff pages
    with tifffile.TiffFile(tif_file) as tif:
        n_pages = len(tif.pages)
        ts = np.zeros(n_pages)
        value = np.zeros(n_pages)
        frame_n = np.zeros(n_pages)
        for i, page in tqdm(enumerate(tif.pages)):

            # extract image description string
            description = page.tags.values()[tag_structure['image_description']].value
            timestamp = float(description.split('\n')[tag_structure['frame_timestamp']].split('=')[-1])  # fetch timestamp in image description
            aux_line = description.split('\n')[tag_structure['auxTrigger0']]
            data = aux_line.split('=')[-1].strip(' [').strip(']').strip(' ]')
            ts[i] = timestamp
            frame_n[i] = i
            if 0 < len(data) < 50:
                try:
                    val = float(data)
                    value[i] = val
                except ValueError:
                    pass
    aux_data['ts'] = ts
    aux_data['value'] = value
    aux_data['frame_n'] = frame_n
    return aux_data, data_reads


In [3]:
def closest_match_indices_sorted(A, B, max_tolerance):
    """
    Return one-to-one closest matches between sorted arrays A and B.

    Parameters
    ----------
    A : array-like
        Sorted array of numbers.
    B : array-like
        Sorted array of numbers.
    max_tolerance : float
        Maximum allowed absolute difference for a match.

    Returns
    -------
    matches : dict of np.ndarray
        Dictionary with keys:
            - "A_index"
            - "B_index"
            - "A_value"
            - "B_value"
            - "abs_difference"
            - "difference"

    unmatched_A_indices : np.ndarray
        Indices of elements in A that were not matched.

    unmatched_B_indices : np.ndarray
        Indices of elements in B that were not matched.
    """
    A = np.asarray(A)
    B = np.asarray(B)

    i = 0
    j = 0

    match_A_indices = []
    match_B_indices = []
    match_A_values = []
    match_B_values = []
    match_abs_differences = []
    match_differences = []

    unmatched_A_indices = []
    matched_B_indices = set()

    while i < len(A):
        a = A[i]

        while j < len(B) and B[j] < a - max_tolerance:
            j += 1

        candidates = []

        if j < len(B) and abs(B[j] - a) <= max_tolerance:
            candidates.append(j)

        # j1 = j
        #
        # while j1 < len(B) and B[j1] <= a + max_tolerance:
        #     if abs(B[j1] - a) <= max_tolerance:
        #         candidates.append(j1)
        #     j1 += 1

        if j > 0 and (j - 1) not in matched_B_indices:
            if abs(B[j - 1] - a) <= max_tolerance:
                candidates.append(j - 1)

        candidates = [idx for idx in candidates if idx not in matched_B_indices]

        if candidates:
            best_j = min(candidates, key=lambda idx: abs(B[idx] - a))

            match_A_indices.append(i)
            match_B_indices.append(best_j)
            match_A_values.append(A[i])
            match_B_values.append(B[best_j])
            match_abs_differences.append(abs(B[best_j] - A[i]))
            match_differences.append(B[best_j] - A[i])
            matched_B_indices.add(best_j)

            if best_j == j:
                j += 1
        else:
            unmatched_A_indices.append(i)

        i += 1

    unmatched_B_indices = np.array(
        [idx for idx in range(len(B)) if idx not in matched_B_indices],
        dtype=int,
    )

    matches = {
        "A_index": np.array(match_A_indices, dtype=int),
        "B_index": np.array(match_B_indices, dtype=int),
        "A_value": np.array(match_A_values, dtype=A.dtype),
        "B_value": np.array(match_B_values, dtype=B.dtype),
        "abs_difference": np.array(match_abs_differences),
        "difference": np.array(match_differences),
    }

    matches = pd.DataFrame(matches)

    return matches, np.array(unmatched_A_indices, dtype=int), unmatched_B_indices

In [4]:
def fix_tsync_time(log_times: np.ndarray):
	skips = - np.where(np.diff(log_times) < 0, np.diff(log_times) - np.median(np.diff(log_times)), 0)
	cs = np.cumsum(skips)
	cs2 = np.hstack((0, cs))
	tsync_time = log_times + cs2
	return tsync_time

In [12]:
pin_sheet_file = '/home/battaglia/src/ofl_2p_analysis/docs/pinSheet_2026.json'
# Path_non_decoded_files = "/data/ofl_2p/A04_day2"
# Path_non_decoded_files = "/data/ofl_2p/A04_day3"
Path_non_decoded_files = "/data/ofl_2p/unity_scanner_totalsync"
# s3d_results_folder = "/Users/fpbattaglia/Dropbox/Data/ofl_2p/s3d-results-00003split_test-std"
Path_output_files = Path("/data/ofl_2p/unity_scanner_totalsync/behavior_preprocessed/")
stats = {}
output_dir = Path(Path_output_files)
output_dir.mkdir(parents=True, exist_ok=True)
b64_files = sorted(str(p) for p in Path(Path_non_decoded_files).glob("*.b64"))
tif_files = sorted(str(p) for p in Path(Path_non_decoded_files).glob("*.tif"))
tsync_file = b64_files[0]
tif_file = tif_files[0]

In [6]:
tsync_session = Path(tsync_file).stem

results_pins = decode_b64_files(Path_non_decoded_files, pin_json_path=pin_sheet_file)

tsync_data = results_pins[tsync_session]


Processing 20260507-171019_550.b64...
/data/ofl_2p/unity_scanner_totalsync/20260507-171019_550.b64
260739 packets, ~4.35 minutes


100%|██████████| 260739/260739 [00:08<00:00, 32408.44it/s]


In [7]:
np.savez(output_dir / f"{tsync_session}_barcode_data.npz", **tsync_data)


In [8]:
tsync_data

{'startTS': array([ 750291992,  750292992,  750293992, ..., 1011147992, 1011148992,
        1011149992], shape=(260739,), dtype=uint32),
 'transmitTS': array([ 750292173,  750293173,  750294173, ..., 1011148173, 1011149173,
        1011150173], shape=(260739,), dtype=uint32),
 'longVar': array([[4294967295,          0,          0, ...,          0,          0,
                181],
        [4294967295,          0,          0, ...,          0,          0,
                181],
        [4294967295,          0,          0, ...,          0,          0,
                181],
        ...,
        [4294967295,          0,          0, ...,          0,          0,
                181],
        [4294967295,          0,          0, ...,          0,          0,
                181],
        [4294967295,          0,          0, ...,          0,          0,
                181]], shape=(260739, 8), dtype=uint32),
 'packetNums': array([ 746290,  746291,  746292, ..., 1007146, 1007147, 1007148],
      

In [9]:

tsync_file = b64_files[0]
tsync_session = Path(tsync_file).stem
tsync_data_00001 = results_pins[tsync_session]
frame_clock = tsync_data['Scanner Frame Clock (Input)'].astype(int)

log_times = tsync_data['startTS'].astype(int)
onsets = np.nonzero(np.diff(frame_clock) == 1)[0]+1 # on the rising edge of the TTL line
stats['max_ts_gap'] = np.max(np.diff(log_times))
print(stats['max_ts_gap'])
stats['gap_locations'] = log_times[np.where(np.diff(log_times) > 30000)]
if len(stats['gap_locations']) > 0:
    warnings.warn(f"There are gaps in the timestamps: {stats['gap_locations']}")
# print(stats['gap_locations'])
tsync_time = fix_tsync_time(log_times)
t_frames = tsync_time[onsets]
t_frames = nap.Ts(t_frames, time_units='us') # time of the frames in Teensy time


121000


/tmp/ipykernel_843477/3080523876.py:12: UserWarning: There are gaps in the timestamps: [1002190992]
  warnings.warn(f"There are gaps in the timestamps: {stats['gap_locations']}")


In [20]:

import re

def normalize_underscores(s: str) -> str:
    return re.sub(r'_+', '_', s).rstrip('_')

tsync_time = fix_tsync_time(tsync_data['startTS'].astype(int))

Path_behavior_npz = Path_output_files / "behavior"
Path_behavior_npz.mkdir(parents=True, exist_ok=True)
for key in tsync_data:
    if tsync_data[key].ndim == 1:
        tsd = nap.Tsd(t=tsync_time, d=tsync_data[key], time_units='us')
        key_name = key.replace(" ", "_")
        key_name = key_name.replace("(", "_").replace(")", "_")
        key_name = normalize_underscores(key_name)
        tsd.save(Path_behavior_npz / f"{key_name}.npz")
    elif tsync_data[key].ndim == 2:
        tsd = nap.TsdFrame(t=tsync_time, d=tsync_data[key], time_units='us')



In [21]:
F = nap.load_file("/data/ofl_2p/20251118/results/behavior_sync/20251118-152602_925_F_sync.npz")

In [22]:
F

Time (s)    0    1    2    3    4    ...
----------  ---  ---  ---  ---  ---  -----
dtype: float64, shape: (0, 743)

In [11]:
aux_data, data_reads = extract_barcode_from_tif(tif_file)

0it [00:00, ?it/s]

In [12]:
aux_data_high = np.nonzero(aux_data['value'])[0]
aux_data_high_ts = aux_data['ts'][aux_data_high]
aux_data_high_val = aux_data['value'][aux_data_high]
aux_barcode_ts = nap.Ts(aux_data_high_val, time_units='s')

aux_data_high_frame_n = aux_data['frame_n'][aux_data_high]
# aux_data_high_frame_n = aux_data_high_frame_n[aux_data_high_ts > 208.5]
# aux_data_high_ts = aux_data_high_ts[aux_data_high_ts > 208.5]

In [13]:
has_barcode  = 'Barcode (Scanner)' in  tsync_data and len(aux_barcode_ts) > 0
stats['has_barcode'] = has_barcode

In [14]:

if has_barcode:
    tsync_barcode = tsync_data['Barcode (Scanner)'].astype(int)
    tsync_barcode_rising_edge = np.nonzero(np.diff(tsync_barcode) > 0)[0]+1
    tsync_barcode_ts = nap.Ts(tsync_time[tsync_barcode_rising_edge], time_units='us')

    barcode_group = nap.TsGroup({0: aux_barcode_ts, 1: tsync_barcode_ts})
    crosscorrs = nap.compute_crosscorrelogram(group=barcode_group, time_units = 'ms', windowsize=2000000, binsize = 1 )
    shift = crosscorrs.idxmax().iloc[0]
    stats['barcode_shift'] = shift
    # print(stats['barcode_shift'])
    matches, unmatched_aux, unmatched_tsync = closest_match_indices_sorted(aux_data['ts']+shift, t_frames.t, max_tolerance=0.025)
    stats['barcode_frame_matches'] = matches
    frames_time_idx = nap.Tsd(t = matches['B_value'].to_numpy(),d = matches['A_index'].to_numpy(), time_units='s')
else:
    warnings.warn("No barcode detected in the tif file, frame/time alignment will be done by assuming that the first Scanner frame clock pulse corresponds to the first tif frame")
    if len(stats['gap_locations']) > 0:
        warnings.warn("With no barcode, and gaps in the timestamps, alignment will be done up to the first gap")
        ep = nap.IntervalSet(start=0, end=stats['gap_locations'][0], time_units='us')
        t_frames = t_frames.restrict(ep)
    frames_time_idx = nap.Tsd(t = t_frames.t, d = np.arange(len(t_frames)), time_units='s')

frames_time_idx.save(output_dir / f"{tsync_session}_frames_time_idx.npz")
with open(output_dir / f"{tsync_session}_behavior_sync_stats.pkl", "wb") as f:
    pickle.dump(stats, f)

In [15]:
frames_time_idx

Time (s)
-----------  ----
770.016992      0
770.050992      1
770.083992      2
770.117992      3
770.150992      4
770.183992      5
770.217992      6
...
1003.993992  7008
1004.027992  7009
1004.060992  7010
1004.093992  7011
1004.127992  7012
1004.160992  7013
1004.194992  7014
dtype: int64, shape: (7012,)

In [16]:
frames2 = nap.load_file(Path('/data/ofl_2p/unity_scanner_totalsync/results_script') / f"{tsync_session}_frames_time_idx.npz")

In [17]:
frames2

Time (s)
-----------  ---
770.016992     0
770.584992     1
770.784992     2
770.984992     3
771.285992     4
771.485992     5
772.186992     6
...
1002.057992  786
1002.491992  787
1002.791992  788
1003.292992  789
1003.593992  790
1003.793992  791
1004.093992  792
dtype: int64, shape: (793,)